# 04 — The estimator & its parameters

*Notebook 4 of 5 — Decision Trees*

You have built a tree by hand three times over — splitting, growing, pruning. Now meet the real
estimator, `DecisionTreeClassifier`, and the dials that control it. We confirm it is the same
mechanism, walk its knobs, and surface the one property — high variance — that the whole back half of
the course exists to fix.

**Prerequisites**

- Notebooks 01–03 — impurity and the best split, growing and reading a tree, overfitting and pruning.
- Module 00 — cross-validation (NB 10) and the train/test split (NB 04).
- Chapter 01 (the scale trap, NB 2) and Chapter 03 (standardization) — the extra work trees turn out
  not to need.

**What you'll be able to do**

- Drive `DecisionTreeClassifier`'s main dials and read what each one does.
- See why a single tree is high-variance — and why that motivates the next five chapters.
- Use trees' native handling of feature scale, more than two classes, and missing values.
- Read feature importance (and its caveat), and tune honestly with `GridSearchCV`.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import datasets, viz
from ml_course.colors import COLORS

np.random.seed(0)
viz.use_course_style()
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# Penguins (binary, raw mm) for the parity check and scale-invariance.
X, y = datasets.penguins_xy()
y_bin = (y == "Gentoo").astype(int).to_numpy()

# make_moons for the knobs and the variance study.
Xm, ym = make_moons(n_samples=300, noise=0.30, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.30, random_state=0, stratify=ym)

# penguins_full (3 species, some missing values) for native multiclass/NaN and importance.
full = datasets.load_penguins_full()
num_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
Xf, yf = full[num_features], full["species"]
print(
    f"penguins {len(y_bin)}, moons train {len(ytr)}, "
    f"penguins_full {len(yf)} ({yf.nunique()} species)"
)

## From by-hand to the estimator

Across the last three notebooks we built a decision tree's machinery ourselves: impurity and the best
split (NB 01), recursive growth and reading the tree (NB 02), overfitting and pruning (NB 03).
`DecisionTreeClassifier` is exactly that machinery behind a clean API. This notebook walks its dials —
and then spends its energy on the property that matters most for what comes next: a single tree is
**unstable**.

In [ ]:
# The by-hand depth-2 rule from NB 02: root flipper <= 206, then bill <= 47.20 / 40.85.
flip = X["flipper_length_mm"].to_numpy()
bill = X["bill_length_mm"].to_numpy()
byhand = np.where(flip <= 206.0, np.where(bill <= 47.20, 0, 1), np.where(bill <= 40.85, 0, 1))

sklearn_pred = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y_bin).predict(X)
print("by-hand == DecisionTreeClassifier(max_depth=2):", np.array_equal(byhand, sklearn_pred))
print(f"training accuracy: {(byhand == y_bin).mean():.4f}")

### Read the result

Same splits, same predictions, the same 0.9964 training accuracy we computed by hand in Notebook 02.
The library is not a black box — it is the mechanism we already understand. With that settled, we can
turn its dials with confidence.

## The dials that control complexity

A decision tree has a handful of knobs, and most of them move the same trade-off — how closely the
tree fits the training data. `max_depth` and `ccp_alpha` are the two we already met (cap the depth;
prune by cost-complexity). `min_samples_leaf` is a third: refuse to make any leaf smaller than *m*
points, so the tree cannot carve a box around a single example. `criterion` chooses the impurity
measure (Gini or entropy). Let us see `min_samples_leaf` in action.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, m in [(ax1, 1), (ax2, 5)]:
    tree = DecisionTreeClassifier(min_samples_leaf=m, random_state=0).fit(Xtr, ytr)
    test = tree.score(Xte, yte)
    viz.plot_decision_boundary(tree, Xtr, ytr, ax=ax)
    ax.set_title(f"min_samples_leaf = {m} — {tree.get_n_leaves()} leaves, test {test:.3f}")
plt.show()

print("min_samples_leaf :  leaves   test")
for m in [1, 5, 20, 50]:
    tree = DecisionTreeClassifier(min_samples_leaf=m, random_state=0).fit(Xtr, ytr)
    print(f"  {m:2d}              :   {tree.get_n_leaves():2d}     {tree.score(Xte, yte):.3f}")

### Read the figure

With `min_samples_leaf = 1` (left) the tree may isolate single points — the jagged, 23-leaf boundary
we know overfits (test 0.878). Raise the floor to 5 (right) and it can no longer build a fence around
one example: the boundary smooths to 16 leaves and test rises to 0.933. The printed table continues
the story — push the floor to 20 and the tree starts to underfit (0.800). Like `max_depth` and
`ccp_alpha`, `min_samples_leaf` is a complexity handle, and you choose it by cross-validation.

In [ ]:
for crit in ["gini", "entropy", "log_loss"]:
    clf = DecisionTreeClassifier(criterion=crit, random_state=0)
    print(f"criterion {crit:8s}: CV {cross_val_score(clf, Xtr, ytr, cv=cv).mean():.4f}")

for mf in [None, 1]:
    clf = DecisionTreeClassifier(max_features=mf, random_state=0)
    print(f"max_features {str(mf):4s}: CV {cross_val_score(clf, Xtr, ytr, cv=cv).mean():.4f}")

### Read the result

Gini, entropy and log-loss land within a hair of each other — on this data the choice of impurity
measure barely matters, which is the usual story. Gini is scikit-learn's default mostly because it
skips the logarithm and is a touch cheaper to compute, not because it scores higher.

`max_features` is different. It tells the tree to consider only a random subset of features at each
split; with one feature it scores a little *worse* as a single tree (0.886 vs 0.910). That looks like
a loss — but injecting randomness so that many trees disagree is exactly the ingredient a **random
forest** averages over (Chapter 06). `class_weight` (reweight the classes) is the knob for class
imbalance; we use it in the next notebook. And `max_depth` / `ccp_alpha` are Notebook 03's dials.

## A single tree is high-variance

Here is the decision tree's defining weakness. Change the training data a little — even only resample
it — and the greedy, top-down construction can pick different splits and grow a **different tree**,
with a different boundary. A model whose answer swings with the sample is called **high-variance**.
Let us see it directly: fit a tree on two different resamples of the same data.

In [ ]:
def grid_pred(model):
    xx, yy = np.meshgrid(
        np.linspace(Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5, 150),
        np.linspace(Xm[:, 1].min() - 0.5, Xm[:, 1].max() + 0.5, 150),
    )
    return model.predict(np.c_[xx.ravel(), yy.ravel()])


# Two example bootstrap resamples (same point cloud shown, only the boundary differs).
rng_fig = np.random.default_rng(0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, label in [(ax1, "resample 1"), (ax2, "resample 2")]:
    idx = rng_fig.integers(0, len(Xtr), len(Xtr))
    tree = DecisionTreeClassifier(random_state=0).fit(Xtr[idx], ytr[idx])
    viz.plot_decision_boundary(tree, Xtr, ytr, ax=ax)
    ax.set_title(f"{label} — {tree.get_n_leaves()} leaves, test {tree.score(Xte, yte):.3f}")
plt.show()

# Variance across 20 resamples, pinned recipe: default_rng(0), random_state=0, 150x150 grid.
rng = np.random.default_rng(0)
for tag, depth in [("full", None), ("depth-3", 3)]:
    preds, accs = [], []
    for _ in range(20):
        idx = rng.integers(0, len(Xtr), len(Xtr))
        tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xtr[idx], ytr[idx])
        preds.append(grid_pred(tree))
        accs.append(tree.score(Xte, yte))
    preds = np.array(preds)
    disagree = [np.mean(preds[i] != preds[j]) for i, j in itertools.combinations(range(20), 2)]
    print(f"{tag:7s}: test std {np.std(accs):.3f}, region disagreement {np.mean(disagree):.1%}")

### Read the figure

Same data-generating process, two bootstrap resamples, two visibly different boundaries — the jagged
regions land in different places, and the held-out accuracy swings from 0.900 to 0.833 on the very same
test set, even though both trees happen to grow 17 leaves. Same size, different shape, different score:
that is variance. The printed numbers put a size on it: across 20 resamples, two unlimited trees
disagree on about 6.3% of the plane; shallow (depth-3) trees are steadier (5.6%) but weaker. This
instability is *the* reason the next chapters exist:
average many trees over many resamples (bagging, then **random forests**, Chapter 06) or add them in
sequence to fix each other's mistakes (boosting, Chapters 07–10), and the variance falls.

## What trees handle without extra machinery

Three chores you did for earlier methods, a tree skips. **Scale:** k-NN needed standardizing (the
scale trap) and logistic regression compared weights only after scaling — a tree splits on thresholds,
so the scale of a feature is irrelevant. **More than two classes** and **missing values**: a tree
handles both directly, with no one-vs-rest wrapper and no imputation step. Let us check all three.

In [ ]:
# Scale-invariance: a tree on raw vs standardized penguins is the same tree.
Xstd = (X - X.mean()) / X.std()
raw_tree = DecisionTreeClassifier(random_state=0).fit(X, y_bin)
std_tree = DecisionTreeClassifier(random_state=0).fit(Xstd, y_bin)
identical = np.array_equal(raw_tree.predict(X), std_tree.predict(Xstd))
print("raw vs standardized -- identical predictions:", identical)
cv_raw = cross_val_score(DecisionTreeClassifier(random_state=0), X, y_bin, cv=cv).mean()
cv_std = cross_val_score(DecisionTreeClassifier(random_state=0), Xstd, y_bin, cv=cv).mean()
print(f"CV  raw {cv_raw:.4f}  ==  standardized {cv_std:.4f}")

# Native multiclass + missing values: penguins_full, 3 species, NaN rows kept (no imputation).
n_missing = int(Xf.isna().any(axis=1).sum())
cv_full = cross_val_score(DecisionTreeClassifier(random_state=0), Xf, yf, cv=cv).mean()
print(f"penguins_full: 3 species, {n_missing} rows with missing values kept  ->  CV {cv_full:.4f}")

### Read the result

The raw tree and the standardized tree make **identical** predictions, with identical cross-validation
accuracy — a split asks "is x ≤ t?", so rescaling a feature only moves the threshold and leaves the
partition unchanged. (Standardizing the whole dataset before cross-validating would leak for a
scale-sensitive model — module 00's warning — but for a tree it provably changes nothing, which is the
point.) The scale trap of Chapter 01 and the `StandardScaler` step of Chapter 03 have no purchase on a
tree. And on the full penguin data — three species, with a couple of rows missing
measurements — the tree trains directly to 0.95 cross-validated accuracy: no one-vs-rest, no
imputation. The one thing scikit-learn still needs is **numbers** — a string category like "island"
must be encoded first (CART's theory handles categories; the implementation wants numeric input).

In [ ]:
tree_full = DecisionTreeClassifier(random_state=0).fit(Xf, yf)
importance = pd.Series(tree_full.feature_importances_, index=num_features).sort_values()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(
    importance.index,
    importance.to_numpy(),
    color=COLORS["class_a"],
    edgecolor=COLORS["text"],
    linewidth=0.6,
)
for i, value in enumerate(importance.to_numpy()):
    ax.text(value, i, f" {value:.2f}", va="center", color=COLORS["text"])
ax.set_xlabel("Gini feature importance")
plt.show()

### Read the figure

Feature importance measures how much each feature reduced impurity across all the tree's splits, added
up. On the penguins, the size measurements dominate — flipper and bill length do almost all the work,
body mass barely any. That reads sensibly. But a caveat travels with this number: **Gini importance is
biased toward features with many distinct values** (continuous or high-cardinality ones), because they
offer more candidate thresholds and so more chances to look useful (Strobl et al., 2007). The honest
cross-check is **permutation importance** — shuffle a feature and watch how much accuracy drops —
which we use in the next notebook.

In [ ]:
grid = {
    "max_depth": [2, 3, 4, 5, 6, None],
    "min_samples_leaf": [1, 5, 10],
    "criterion": ["gini", "entropy"],
    "ccp_alpha": [0.0, 0.005, 0.01],
}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=cv).fit(Xtr, ytr)
print("best params:", search.best_params_)
print(f"best CV: {search.best_score_:.4f}   sealed test: {search.score(Xte, yte):.4f}")

### Read the result

`GridSearchCV` searched the dials on the training set with cross-validation and never touched the test
set until the end. It landed on `max_depth = 6` — the very depth Notebook 03 chose by hand — for a
sealed-test accuracy of 0.889. This is the module-00 discipline (tune by cross-validation on the
training set, read the test once), now run across four knobs at once instead of one.

## Your turn

1. **A bigger leaf floor.** Set `min_samples_leaf=10` on the moons tree and report its leaf count and
   test accuracy. Where does it sit between the 5 and the 20 from the table?
2. **Units don't matter.** Refit the penguin tree with `flipper_length` converted to centimetres
   (divide by 10) and `bill_length` left in millimetres. Confirm the predictions are unchanged — and
   say why in one sentence.
3. **The seed of a forest.** Fit `max_features=1` five times with `random_state` 0…4 and describe how
   much the decision boundary moves. That movement is what a random forest will average away (Chapter
   06).

## What you built

- Drove `DecisionTreeClassifier`'s dials: `max_depth`, `min_samples_leaf`, `ccp_alpha`, `criterion`
  (and met `max_features` and `class_weight`).
- Saw, and measured, a single tree's **high variance** — the precise reason the next five chapters
  average or add trees.
- Used trees' **native** handling of feature scale, multiple classes, and missing values — work you
  did by hand for k-NN and logistic regression.
- Read **Gini feature importance** and its bias, and tuned honestly with **`GridSearchCV`**.

**Vocabulary:** `min_samples_leaf` · `criterion` · `max_features` · `class_weight` · variance /
instability · scale-invariance · feature importance · permutation importance · `GridSearchCV`.

## Going further (optional)

- **Randomized trees.** `max_features` < n is the heart of a random forest: each tree sees a random
  feature subset, so the trees decorrelate and their average is far steadier than any one (Chapter 06).
- **Two importances.** Impurity (Gini) importance is fast but biased; permutation importance is slower
  but model-agnostic and honest. Report the second when a feature's importance really matters.
- **Categoricals.** CART can split on categories directly; scikit-learn's implementation needs them
  encoded to numbers first (one-hot or ordinal).

## References

- Breiman, Friedman, Olshen & Stone (1984). *Classification and Regression Trees (CART).* Wadsworth.
- Strobl, C., Boulesteix, A.-L., Zeileis, A. & Hothorn, T. (2007). *Bias in random forest variable
  importance measures.* BMC Bioinformatics 8:25. DOI: 10.1186/1471-2105-8-25.
- Breiman, L. (1996). *Bagging predictors.* Machine Learning 24, 123–140. DOI: 10.1007/BF00058655.
- Hastie, Tibshirani & Friedman (2009). *The Elements of Statistical Learning*, §9.2.
  DOI: 10.1007/978-0-387-84858-7.
- James, Witten, Hastie & Tibshirani (2021). *An Introduction to Statistical Learning*, §8.1.
  DOI: 10.1007/978-1-0716-1418-1.

*Previous: 03 — Overfitting & pruning · Next: 05 — A demanding case: breast cancer.*